# Dobot CR3A: Newton ModelBuilder and Simple Motion

CR3A URDF를 Newton `ModelBuilder`로 불러오고, Viser에서 확인한 뒤 간단한 관절 동작을 재생합니다.

현재는 joint space에서 직접 움직이는 FK 기반 움직임이다.  



## 1. Setup and Imports

Newton과 CR3A 로더에 필요한 최소 모듈만 불러옵니다.


In [5]:
import os
import time
from pathlib import Path

import numpy as np
import warp as wp
import newton

wp.config.quiet = True
wp.set_device("cpu")


## 2. CR3A URDF Utility

URDF의 예전 빌드 머신 경로를 현재 `assets` 폴더로 바꾼 임시 복사본을 만듭니다. 원본 URDF는 수정하지 않습니다.


In [6]:
def prepare_cr3a_urdf():
    default_path = (
        Path.home()
        / "workspaces/movensys_ws/src/movensys-manipulator"
        / "movensys_manipulator_description/urdf/dobot_cr3a"
        / "movensys_manipulator.urdf"
    )
    source_urdf = Path(os.environ.get("CR3A_URDF", default_path)).expanduser().resolve()
    assets_dir = source_urdf.parent / "assets"

    if not source_urdf.is_file():
        raise FileNotFoundError(
            f"CR3A URDF not found: {source_urdf}\n"
            "Set the CR3A_URDF environment variable if it is stored elsewhere."
        )
    if not assets_dir.is_dir():
        raise FileNotFoundError(f"CR3A assets directory not found: {assets_dir}")

    old_asset_uri = (
        "file:///home/hehe-jazzy/workspaces/movensys_ws/install/"
        "movensys_manipulator_description/share/"
        "movensys_manipulator_description/urdf/dobot_cr3a/assets/"
    )
    patched_text = source_urdf.read_text(encoding="utf-8").replace(
        old_asset_uri, assets_dir.as_posix() + "/"
    )

    patched_urdf = Path("/tmp/dobot_cr3a_newton.urdf")
    patched_urdf.write_text(patched_text, encoding="utf-8")
    return patched_urdf


## 3. Build the CR3A Model

`ModelBuilder`에 CR3A URDF를 추가하고 시뮬레이션/시각화에 사용할 모델과 상태를 만듭니다.  

`add_urdf()`라는 함수는 이름 그대로 URDF의 파일을 불러온다. 하는 역할은 다음과 같다.
- URDF의 link를 Newton body로 생성
- URDF의 joint를 Newton joint로 생성
- joint 타입, 축 방향, 위치 제한, 속도 제한, effort 제한 등을 읽음
- 질량과 관성, collision/visual geometry를 모델에 등록
- URDF에 drive gain 정보가 있으면 일부 joint drive 설정을 반영할 수 있음  

모터 객체, PID 제어기, 모터 토크 계산 로직, 전원/감속기/엔코더 모델, 각 관절에 대한 제어 명령 생성 루프는 자동으로 생성되지 않는다.


In [7]:
patched_urdf = prepare_cr3a_urdf()

builder = newton.ModelBuilder()
builder.add_urdf(
    str(patched_urdf),
    floating=False,
    enable_self_collisions=False,
    parse_visuals_as_colliders=False,
)

model = builder.finalize()
state = model.state()
newton.eval_fk(model, model.joint_q, model.joint_qd, state)

print("Newton device:", model.device)
print("Patched URDF:", patched_urdf)
print(
    f"CR3A loaded: {model.body_count} bodies, "
    f"{model.joint_count} joints, {model.joint_coord_count} coordinates, "
    f"{model.shape_count} shapes"
)


Newton device: cpu
Patched URDF: /tmp/dobot_cr3a_newton.urdf
CR3A loaded: 14 bodies, 14 joints, 8 coordinates, 26 shapes


## 4. Preview CR3A with Viser

초기 자세를 Viser에 띄웁니다. 셀을 다시 실행하면 이전 CR3A viewer를 닫고 새로 엽니다.


In [ ]:
previous_viewer = globals().get("cr3a_viewer")
if previous_viewer is not None:
    previous_viewer.close()

cr3a_viewer = newton.viewer.ViewerViser(
    port=8090,
    label="CR3A Robot",
    verbose=True,
)
cr3a_viewer.set_model(model)
cr3a_viewer.begin_frame(0.0)
cr3a_viewer.log_state(state)
cr3a_viewer.end_frame()

print("Open the CR3A viewer at:", cr3a_viewer.url)
cr3a_viewer


(viser) Server stopped

╭────── viser (listening *:8090) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8090   │
│   Websocket │ ws://localhost:8090     │
│             ╵                         │
╰───────────────────────────────────────╯

Viser server running at: http://localhost:8090/
Open the CR3A viewer at: http://localhost:8090/


(viser) Connection opened (0, 1 total), 36 persistent messages

## 5. Play a Simple CR3A Motion

먼저 4번 셀에서 연 Viser 화면을 띄워 둔 채 이 셀을 실행하세요. 1·2·4번 관절이 실제 시간 속도로 두 번 왕복합니다. 물리 제어기를 붙이기 전 모델/관절 방향을 확인하기 위한 가장 단순한 kinematic 예제입니다.


In [ ]:
fps = 30
cycle_duration = 4.0
duration = 2.0 * cycle_duration
num_frames = int(fps * duration)

home_q = model.joint_q.numpy().astype(np.float32)
animated_q = wp.clone(model.joint_q)

if home_q.size < 6:
    raise RuntimeError(f"Expected at least 6 CR3A joint coordinates, got {home_q.size}")

print("Playing CR3A motion in the Viser window...")
playback_start = time.perf_counter()

for frame in range(num_frames + 1):
    t = frame / fps
    phase = 2.0 * np.pi * t / cycle_duration

    q = home_q.copy()
    q[0] += 0.45 * np.sin(phase)
    q[1] += 0.25 * np.sin(phase + 0.5 * np.pi)
    q[3] += 0.35 * np.sin(phase)

    animated_q.assign(q)
    newton.eval_fk(model, animated_q, model.joint_qd, state)

    cr3a_viewer.begin_frame(t)
    cr3a_viewer.log_state(state)
    cr3a_viewer.end_frame()

    # 프레임을 한꺼번에 보내지 않고 벽시계 시간에 맞춰 실시간 재생합니다.
    next_frame_time = playback_start + (frame + 1) / fps
    time.sleep(max(0.0, next_frame_time - time.perf_counter()))

print(f"Playback finished ({duration:.1f} s). Run this cell again to replay.")
cr3a_viewer


Playing CR3A motion in the Viser window...


(viser) Connection closed (3, 1 total)

Playback finished (8.0 s). Run this cell again to replay.


(viser) Connection opened (4, 2 total), 52 persistent messages

5번 셀에서 하는 건 IK가 아님. 그냥 관절각 q[0], q[1], q[3]를 사인파로 직접 바꾼 뒤 newton.eval_fk(...)로 결과 자세만 다시 계산해서 Viser에 보여주는 방식. 코드상으로도 EEF 목표를 주는 부분이 없고, 관절각을 직접 넣고 있음

IK 계산
- 입력: EEF 목표 위치/자세
- 출력: 그 자세를 만들 관절각 q
이 계산에는 링크 길이, 조인트 축, 조인트 제한 같은 URDF 정보가 중요하지, 모터 모델이 꼭 필요하진 않습니다.

모터/액추에이터가 필요한 건 그 다음 단계
- IK: 목표 EEF pose -> 목표 관절각 계산
- 제어기: 목표 관절각 -> 각 관절에 줄 토크/속도/전류/position command 계산
- 물리/실기기: 그 명령으로 실제로 움직임

즉 지금 노트북은:

- URDF 로드
- FK로 자세 계산
- 관절각을 사람이 직접 지정해서 시각화까지만 한 상태입니다.

아직 없는 것은:

- EEF target 입력
- IK solver
- IK 결과를 시간에 따라 부드럽게 보내는 trajectory 생성
- 각 관절 actuator/servo 제어기